# r2 tables

In [20]:
import numpy as np
import pandas as pd
import polpo.preprocessing.dict as ppdict
import polpo.preprocessing.pd as ppd
from polpo.model_eval import (
    MeshEuclideanR2Score,
    MeshR2Score,
    MultiEvaluator,
    OlsPValues,
    PcaEvaluator,
    R2Score,
    ReconstructionError,
    ResultsExtender,
    VertexReconstructionError,
    collect_obj_regr_eval_results,
)
from polpo.models import ObjectRegressor, SupervisedEmbeddingRegressor
from polpo.preprocessing import PartiallyInitializedStep
from polpo.preprocessing.learning import DictsToXY
from polpo.preprocessing.load.pregnancy import (
    DenseMaternalCsvDataLoader,
    DenseMaternalMeshLoader,
)
from polpo.preprocessing.mesh.conversion import PvFromData
from polpo.preprocessing.mesh.io import FreeSurferReader
from polpo.preprocessing.mesh.registration import PvAlign
from polpo.sklearn.adapter import AdapterPipeline, EvaluatedModel
from polpo.sklearn.mesh import BiMeshesToVertices
from polpo.sklearn.np import BiFlattenButFirst
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import r2_score
from scipy.stats import f

subject_id = "01"
pilot = subject_id == "01"

# List of structures to loop through
structures = [
    "Hipp",
    "Amyg",
    "Thal",
    "Caud",
    "Puta",
    "Pall",
    "Accu",
]

# Prepare an empty list to collect results
results_list = []

# Loop through each structure and hemisphere
for struct in structures:
    for left in [True, False]:
        try:
            # Load data
            csv_loader = DenseMaternalCsvDataLoader(pilot=pilot, subject_id=subject_id)
            df = csv_loader()

            # Preprocess predictor
            session_selector = ppd.DfIsInFilter("stage", ["post"], negate=True)
            predictor_selector = (
                session_selector + ppd.ColumnsSelector("gestWeek") + ppd.SeriesToDict()
            )
            x_dict = predictor_selector(df)
            # print(x_dict)
            train_dict = {k: v for k, v in x_dict.items() if 1 <= k <= 16}
            test_dict = {k: v for k, v in x_dict.items() if 17 <= k <= 19}

            # Load and preprocess meshes
            mesh_loader = DenseMaternalMeshLoader(
                subject_id=subject_id,
                as_dict=True,
                left=left,
                struct=struct,
                derivative="enigma",
            )
            mesh_reader = ppdict.DictMap(FreeSurferReader() + PvFromData())

            prep_pipe = PartiallyInitializedStep(
                Step=lambda **kwargs: ppdict.DictMap(PvAlign(**kwargs)),
                _target=lambda meshes: meshes[list(meshes.keys())[0]],
                max_iterations=500,
            )

            mesh_pipe = mesh_loader + mesh_reader + prep_pipe
            meshes = mesh_pipe()

            # Model and pipeline setup
            objs2y = AdapterPipeline(
                steps=[
                    BiMeshesToVertices(index=0),
                    FunctionTransformer(func=np.stack),
                    BiFlattenButFirst(),
                ]
            )

            model = SupervisedEmbeddingRegressor(
                EvaluatedModel(
                    PLSRegression(n_components=2),
                    MultiEvaluator(
                        [
                            ReconstructionError(),
                            VertexReconstructionError(prefix="vertex"),
                        ]
                    ),
                ),
                EvaluatedModel(
                    LinearRegression(),
                    MultiEvaluator([OlsPValues(), R2Score()]),
                ),
            )

            obj_model = EvaluatedModel(
                ObjectRegressor(model, objs2y),
                MultiEvaluator(
                    [MeshEuclideanR2Score(), MeshR2Score()],
                    extender=ResultsExtender(),
                ),
            )

            # Create dataset
            dataset_pipe = DictsToXY()
            X, meshes_ = dataset_pipe((x_dict, meshes))
            X_train, meshes_train = dataset_pipe((train_dict, meshes))
            X_test, meshes_test = dataset_pipe((test_dict, meshes))

            # Fit and evaluate
            # obj_model.fit(X, meshes_)
            obj_model.fit(X_train, meshes_train)
            predictions_test = obj_model.predict(X_test)

            # Evaluate on train and test sets
            eval_results_train = collect_obj_regr_eval_results(obj_model)

            # Flatten true and predicted meshes
            true_flat = np.stack([m.points.flatten() for m in meshes_test])
            pred_flat = np.stack([m.points.flatten() for m in predictions_test])

            # Compute global R² for test data
            r2_test = r2_score(true_flat.flatten(), pred_flat.flatten())

            # Extract p-values and R² from "regr-encoder"
            regr_encoder = eval_results["regr-encoder"]
            pvalues = eval_results_train["regr-regr"]["pvals"]
            r2_train = eval_results_train["obj_regr"]["featurewise_r2-max"]
            r2_test = r2_test

            # Compute p-value for H0: R² = 0
            n = len(X_test)  # number of samples in test set
            k = 1  # single predictor (gestWeek)
            F_stat = (r2_test / k) / ((1 - r2_test) / (n - k - 1))
            p_value_r2 = 1 - f.cdf(F_stat, k, n - k - 1)

            # Store results
            results_list.append(
                {
                    "structure": struct,
                    "left": left,
                    "p-values": pvalues,
                    "r2-train": r2_train,
                    "r2-test": r2_test,
                    "r2-test-p-value": p_value_r2,
                }
            )

        except Exception as e:
            print(f"Error processing {struct} {'left' if left else 'right'}: {e}")
            results_list.append(
                {
                    "structure": struct,
                    "left": left,
                    "p-values": np.nan,
                    "r2-train": np.nan,
                    "r2-test": np.nan,
                    "r2-test-p-value": np.nan,
                }
            )


# Create a DataFrame for easier viewing
results_df = pd.DataFrame(results_list)

INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Da

In [21]:
# Display the table
import tabulate

print(tabulate.tabulate(results_df, headers="keys", tablefmt="psql"))

+----+-------------+--------+--------------------+------------+-----------+-------------------+
|    | structure   | left   | p-values           |   r2-train |   r2-test |   r2-test-p-value |
|----+-------------+--------+--------------------+------------+-----------+-------------------|
|  0 | Hipp        | True   | [[1.48918646e-06]  |   0.782678 |  0.999447 |        0.014969   |
|    |             |        |  [2.86851788e-01]] |            |           |                   |
|  1 | Hipp        | False  | [[1.94823137e-06]  |   0.807834 |  0.999668 |        0.0116013  |
|    |             |        |  [2.21610029e-01]] |            |           |                   |
|  2 | Amyg        | True   | [[0.00122685]      |   0.790701 |  0.990919 |        0.0607592  |
|    |             |        |  [0.01806117]]     |            |           |                   |
|  3 | Amyg        | False  | [[0.00040494]      |   0.67042  |  0.999886 |        0.00679144 |
|    |             |        |  [0.077311

In [22]:
# GOING TO TRY TO MAKE A TABLE FOR VOLUMES

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy.stats import f

results_list = []

for struct in structures:
    for left in [True, False]:
        try:
            # Load data
            csv_loader = DenseMaternalCsvDataLoader(pilot=pilot, subject_id=subject_id)
            df = csv_loader()

            # Preprocess predictor
            session_selector = ppd.DfIsInFilter("stage", ["post"], negate=True)
            predictor_selector = (
                session_selector + ppd.ColumnsSelector("gestWeek") + ppd.SeriesToDict()
            )
            x_dict = predictor_selector(df)

            # Load and preprocess meshes
            mesh_loader = DenseMaternalMeshLoader(
                subject_id=subject_id,
                as_dict=True,
                left=left,
                struct=struct,
                derivative="enigma",
            )
            mesh_reader = ppdict.DictMap(FreeSurferReader() + PvFromData())

            prep_pipe = PartiallyInitializedStep(
                Step=lambda **kwargs: ppdict.DictMap(PvAlign(**kwargs)),
                _target=lambda meshes: meshes[list(meshes.keys())[0]],
                max_iterations=500,
            )

            mesh_pipe = mesh_loader + mesh_reader + prep_pipe
            meshes = mesh_pipe()

            # Create dataset
            dataset_pipe = DictsToXY()
            X_all, meshes_all = dataset_pipe((x_dict, meshes))

            # Sessions 1–16: train, 17–19: test
            train_sessions = list(range(0, 16))
            test_sessions = list(range(16, 19))

            X_train = X_all[train_sessions]
            X_test = X_all[test_sessions]
            meshes_train = [meshes_all[i] for i in train_sessions]
            meshes_test = [meshes_all[i] for i in test_sessions]

            # Compute mesh volumes
            volumes_train = np.array([m.volume() for m in meshes_train])
            volumes_test = np.array([m.volume() for m in meshes_test])

            # Fit simple linear regression: gestWeek -> volume
            regr = LinearRegression()
            regr.fit(X_train, volumes_train)

            # Predict on train and test
            pred_train = regr.predict(X_train)
            pred_test = regr.predict(X_test)

            # Compute R² for train and test
            r2_train = r2_score(volumes_train, pred_train)
            r2_test = r2_score(volumes_test, pred_test)

            # Compute p-value for H0: R² = 0 (test data)
            n = len(X_test)
            k = 1
            F_stat = (r2_test / k) / ((1 - r2_test) / (n - k - 1))
            p_value_r2 = 1 - f.cdf(F_stat, k, n - k - 1)

            # Store results
            results_list.append(
                {
                    "structure": struct,
                    "left": left,
                    "r2-train": r2_train,
                    "r2-test": r2_test,
                    "r2-test-p-value": p_value_r2,
                }
            )

        except Exception as e:
            print(f"Error processing {struct} {'left' if left else 'right'}: {e}")
            results_list.append(
                {
                    "structure": struct,
                    "left": left,
                    "r2-train": np.nan,
                    "r2-test": np.nan,
                    "r2-test-p-value": np.nan,
                }
            )

# Create DataFrame for easier viewing
results_df = pd.DataFrame(results_list)

# Display the results
print(results_df)

INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Error processing Hipp left: 'float' object is not callable
Error processing Hipp right: 'float' object is not callable


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Error processing Amyg left: 'float' object is not callable


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


...

Error processing Puta right: 'float' object is not callable


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Error processing Pall left: 'float' object is not callable


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Error processing Pall right: 'float' object is not callable
Error processing Accu left: 'float' object is not callable
Error processing Accu right: 'float' object is not callable
   structure   left  r2-train  r2-test  r2-test-p-value
0       Hipp   True       NaN      NaN              NaN
1       Hipp  False       NaN      NaN              NaN
2       Amyg   True       NaN      NaN              NaN
3       Amyg  False       NaN      NaN              NaN
4       Thal   True       NaN      NaN              NaN
5       Thal  False       NaN      NaN              NaN
6       Caud   True       NaN      NaN              NaN
7       Caud  False       NaN      NaN              NaN
8       Puta   True       NaN      NaN              NaN
9       Puta  False       NaN      NaN              NaN
10      Pall   True       NaN      NaN              NaN
11      Pall  False       NaN      NaN              NaN
12      Accu   True       NaN      NaN              NaN
13      Accu  False       NaN      Na